# fig data distribution

In [ ]:
from pathlib import Path

from icicle.utils.visualization.style import (
    FIGSIZE,
    get_palette,
    make_fig,
    save_fig,
    set_style,
)
from tqdm import tqdm
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

set_style("manuscript")
palette = get_palette()
OUTPUT_DIR = Path("figures/data_distribution")

In [ ]:
def mw_from_smiles(smiles):
    """Calculate molecular weight from SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Descriptors.MolWt(mol)


def get_scaffold_smiles(smiles):
    """Get Murcko scaffold SMILES from a molecule SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)


def analyze_split(
    spec_file: str,
    split_file: str,
    split_name: str = "test",
    n_random_samples: int = 5,
):
    """
     Analyze a dataset split and generate statistics and histograms.

     Parameters
    -------
     spec_file : str
         Path to the HDF5 spectra file.
     split_file : str
         Path to the TSV split file.
     split_name : str
         Name of split to analyze ('train', 'val', or 'test').
     n_random_samples : int
         Number of random molecules to display with scaffolds.
    """
    # Load splits and get inchi_keys for the desired split
    splits = pd.read_csv(split_file, sep="\t")
    split_inchi_keys = set(
        splits[splits["split"] == split_name]["inchi_key"].values
    )

    print(f"Analyzing {split_name} split from:")
    print(f"  Spectra: {spec_file}")
    print(f"  Splits: {split_file}")
    print(f"  Number of samples in {split_name}: {len(split_inchi_keys)}")
    print("-" * 60)

    # Collect data
    molecular_weights = []
    num_peaks_list = []
    num_peaks_above_01_list = []
    smiles_list = []
    mol_ids = []
    skipped = 0

    with h5py.File(spec_file, "r") as f:
        keys = list(f.keys())
        for key in tqdm(keys, desc="Processing", unit="spectrum"):
            # Skip entries without inchi_key field
            if "inchi_key" not in f[key]:
                skipped += 1
                continue

            inchi_key = f[key]["inchi_key"][()].decode("utf-8")
            if inchi_key not in split_inchi_keys:
                continue

            smiles = f[key]["standardized_smiles"][()].decode("utf-8")
            intensities = f[key]["intensities"][()]

            mw = mw_from_smiles(smiles)
            if mw is None:
                continue

            num_peaks = len(intensities)
            num_peaks_above_01 = np.sum(intensities > 0.1)

            molecular_weights.append(mw)
            num_peaks_list.append(num_peaks)
            num_peaks_above_01_list.append(num_peaks_above_01)
            smiles_list.append(smiles)
            mol_ids.append(key)

    if skipped > 0:
        print(f"  (Skipped {skipped} entries without inchi_key)")

    # Convert to arrays
    molecular_weights = np.array(molecular_weights)
    num_peaks_list = np.array(num_peaks_list)
    num_peaks_above_01_list = np.array(num_peaks_above_01_list)

    # Print statistics
    print(f"\nStatistics for {split_name} split (n={len(molecular_weights)}):")
    print(
        f"  Molecular Weight:      {molecular_weights.mean():.2f} ± {molecular_weights.std():.2f}"
    )
    print(
        f"  Number of Peaks:       {num_peaks_list.mean():.2f} ± {num_peaks_list.std():.2f}"
    )
    print(
        f"  Peaks > 0.1 intensity: {num_peaks_above_01_list.mean():.2f} ± {num_peaks_above_01_list.std():.2f}"
    )

    # Print random molecules with scaffolds
    print(f"\n{n_random_samples} Random Molecules and Scaffolds:")
    print("-" * 60)
    random_indices = np.random.choice(
        len(smiles_list),
        min(n_random_samples, len(smiles_list)),
        replace=False,
    )
    for idx in random_indices:
        smi = smiles_list[idx]
        scaffold = get_scaffold_smiles(smi)
        print(f"  Mol ID: {mol_ids[idx]}")
        print(f"    SMILES:   {smi}")
        print(f"    Scaffold: {scaffold}")
        print(
            f"    MW: {molecular_weights[idx]:.2f}, Peaks: {num_peaks_list[idx]}, Peaks>0.1: {num_peaks_above_01_list[idx]}"
        )
        print()

    # Create histograms (one panel per metric, styled per project convention)
    hist_specs = [
        (molecular_weights, "Molecular Weight (Da)", "mw"),
        (num_peaks_list, "Number of Peaks", "num_peaks"),
        (
            num_peaks_above_01_list,
            "Number of Peaks > 0.1 Intensity",
            "peaks_above_01",
        ),
    ]
    for data, xlabel, stem in hist_specs:
        fig, ax = make_fig("default")
        ax.hist(data, bins=50, alpha=0.7, color=palette[0])
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Count")
        ax.axvline(
            data.mean(),
            color=palette[2],
            linestyle="--",
            label=f"Mean: {data.mean():.1f}",
        )
        ax.legend()
        save_fig(fig, f"{split_name}_{stem}_hist", OUTPUT_DIR)
        plt.show()
        plt.close(fig)

    return {
        "molecular_weights": molecular_weights,
        "num_peaks": num_peaks_list,
        "num_peaks_above_01": num_peaks_above_01_list,
        "smiles": smiles_list,
        "mol_ids": mol_ids,
    }

In [ ]:
# NIST spectra + both splits (scaffold and random)
spec_file_nist = "../../data/NIST2023_GCMS_main/spectra.hdf5"
split_file_nist_scaffold = "../../data/NIST2023_GCMS_main/splits/scaffold_no_xeno_aas_deduplicated.tsv"
split_file_nist_random = (
    "../../data/NIST2023_GCMS_main/splits/random_no_xeno_aas_deduplicated.tsv"
)

In [ ]:
nist_train_stats = analyze_split(
    spec_file=spec_file_nist,
    split_file=split_file_nist_scaffold,
    split_name="train",
    n_random_samples=5,
)
nist_val_stats = analyze_split(
    spec_file=spec_file_nist,
    split_file=split_file_nist_scaffold,
    split_name="val",
    n_random_samples=5,
)

In [ ]:
nist_test_stats = analyze_split(
    spec_file=spec_file_nist,
    split_file=split_file_nist_scaffold,
    split_name="test",
    n_random_samples=5,
)

In [ ]:
nist_train_stats_random = analyze_split(
    spec_file=spec_file_nist,
    split_file=split_file_nist_random,
    split_name="train",
    n_random_samples=5,
)
nist_val_stats_random = analyze_split(
    spec_file=spec_file_nist,
    split_file=split_file_nist_random,
    split_name="val",
    n_random_samples=5,
)
nist_test_stats_random = analyze_split(
    spec_file=spec_file_nist,
    split_file=split_file_nist_random,
    split_name="test",
    n_random_samples=5,
)

In [ ]:
import matplotlib.gridspec as gridspec


def plot_marginal_histograms(
    series_dict: dict,
    x_key: str,
    y_key: str,
    x_label: str = None,
    y_label: str = None,
    figsize: tuple = FIGSIZE["square"],
    bins: int = 30,
    alpha: float = 0.6,
    s: float = 15,
    colors: list = None,
    hist_alpha: float = 0.5,
    density: bool = True,
    legend_loc: str = "upper right",
    legend_bbox: tuple = (1.0, 1.0),
    x_lim: tuple = None,
    y_lim: tuple = None,
    save_name: str = None,
    sample: list = None,
):
    """
     Plot 2D scatter with marginal histograms for multiple series.

     Parameters
    -------
     series_dict : dict
         Dictionary mapping series labels to data dicts.
         Each data dict must have keys matching x_key and y_key.
         Example: {"NIST Train": nist_train_stats, "NIST Test": nist_test_stats}
     x_key : str
         Key to extract x-axis data from each series dict.
     y_key : str
         Key to extract y-axis data from each series dict.
     x_label : str, optional
         Label for x-axis. Defaults to x_key.
     y_label : str, optional
         Label for y-axis. Defaults to y_key.
     figsize : tuple
         Figure size (width, height) — plot area, per FIGSIZE convention.
     bins : int
         Number of bins for histograms.
     alpha : float
         Scatter plot point transparency.
     s : float
         Scatter plot point size.
     colors : list, optional
         List of colors for each series. Defaults to the project palette.
     hist_alpha : float
         Histogram transparency.
     density : bool
         If True, normalize histograms to density.
     legend_loc : str
         Legend location.
     legend_bbox : tuple
         Legend bbox_to_anchor.
     x_lim : tuple, optional
         X-axis limits (min, max).
     y_lim : tuple, optional
         Y-axis limits (min, max).
     save_name : str, optional
         Filename stem to save the figure under OUTPUT_DIR via save_fig.
     sample : list, optional
         Per-series subsample size (None = use all points for that series).

     Returns
    ----
     fig : matplotlib.figure.Figure
     axes : dict
         Dictionary with 'joint', 'marg_x', 'marg_y' axes.
    """
    if colors is None:
        colors = list(get_palette())

    x_label = x_label or x_key
    y_label = y_label or y_key

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = gridspec.GridSpec(
        2,
        2,
        width_ratios=[4, 1],
        height_ratios=[1, 4],
        figure=fig,
        wspace=0.05,
        hspace=0.05,
    )

    ax_joint = fig.add_subplot(gs[1, 0])
    ax_marg_x = fig.add_subplot(gs[0, 0], sharex=ax_joint)
    ax_marg_y = fig.add_subplot(gs[1, 1], sharey=ax_joint)

    for i, (label, data) in enumerate(series_dict.items()):
        if sample and sample[i] is not None:
            df = pd.DataFrame(data)
            data = df.sample(n=sample[i], random_state=42).to_dict(
                orient="list"
            )
        x_data = np.array(data[x_key])
        y_data = np.array(data[y_key])
        color = colors[i % len(colors)]

        ax_joint.scatter(
            x_data,
            y_data,
            s=s,
            alpha=alpha,
            color=color,
            label=f"{label} (n={len(x_data)})",
            edgecolors="none",
        )
        ax_marg_x.hist(
            x_data,
            bins=bins,
            alpha=hist_alpha,
            color=color,
            density=density,
            linewidth=0.5,
        )
        ax_marg_x.axvline(
            x_data.mean(), color=color, linestyle="--", linewidth=1.0
        )
        ax_marg_y.hist(
            y_data,
            bins=bins,
            alpha=hist_alpha,
            color=color,
            density=density,
            orientation="horizontal",
            linewidth=0.5,
        )
        ax_marg_y.axhline(
            y_data.mean(), color=color, linestyle="--", linewidth=1.0
        )

    ax_joint.set_xlabel(x_label)
    ax_joint.set_ylabel(y_label)
    if x_lim:
        ax_joint.set_xlim(x_lim)
    if y_lim:
        ax_joint.set_ylim(y_lim)

    for ax in (ax_marg_x, ax_marg_y):
        ax.tick_params(
            axis="both",
            which="both",
            bottom=False,
            top=False,
            left=False,
            right=False,
            labelbottom=False,
            labelleft=False,
        )
        for spine in ("top", "right", "left", "bottom"):
            ax.spines[spine].set_visible(False)
    ax_marg_x.set_ylabel("")
    ax_marg_y.set_xlabel("")

    ax_joint.legend(loc=legend_loc, bbox_to_anchor=legend_bbox, frameon=False)

    if save_name:
        save_fig(fig, save_name, OUTPUT_DIR)
    plt.show()
    plt.close(fig)

    return fig, {"joint": ax_joint, "marg_x": ax_marg_x, "marg_y": ax_marg_y}

In [ ]:
# Scaffold split: MW vs. num peaks, train/val/test
plot_marginal_histograms(
    series_dict={
        "Train": nist_train_stats,
        "Val": nist_val_stats,
        "Test": nist_test_stats,
    },
    sample=[5000, None, 5000],
    x_key="molecular_weights",
    y_key="num_peaks",
    x_label="Molecular Weight (Da)",
    y_label="Number of Peaks",
    bins=40,
    alpha=0.5,
    s=5,
    colors=[palette[0], palette[3], palette[6]],
    legend_loc="upper left",
    legend_bbox=(0, 1.0),
    save_name="marginal_mw_vs_num_peaks_scaffold",
)

In [ ]:
# Scaffold split: peaks > 0.1 vs. num peaks, train/val/test
# Shared axis limits (num_peaks is the wider range since peaks>0.1 is a subset of it)
_shared_max_scaffold = max(
    nist_train_stats["num_peaks"].max(),
    nist_val_stats["num_peaks"].max(),
    nist_test_stats["num_peaks"].max(),
)
plot_marginal_histograms(
    series_dict={
        "Train": nist_train_stats,
        "Val": nist_val_stats,
        "Test": nist_test_stats,
    },
    sample=[5000, None, 5000],
    x_key="num_peaks_above_01",
    y_key="num_peaks",
    x_label="Number of Peaks > 0.1",
    y_label="Number of Peaks",
    bins=40,
    alpha=0.5,
    s=5,
    colors=[palette[0], palette[3], palette[6]],
    legend_loc="upper left",
    legend_bbox=(0, 1.0),
    x_lim=(0, _shared_max_scaffold),
    y_lim=(0, _shared_max_scaffold),
    save_name="marginal_peaks_above_01_vs_num_peaks_scaffold",
)

In [ ]:
# Random split: MW vs. num peaks, train/val/test
plot_marginal_histograms(
    series_dict={
        "Train": nist_train_stats_random,
        "Val": nist_val_stats_random,
        "Test": nist_test_stats_random,
    },
    sample=[5000, None, 5000],
    x_key="molecular_weights",
    y_key="num_peaks",
    x_label="Molecular Weight (Da)",
    y_label="Number of Peaks",
    bins=40,
    alpha=0.5,
    s=5,
    colors=[palette[0], palette[3], palette[6]],
    legend_loc="upper left",
    legend_bbox=(0, 1.0),
    save_name="marginal_mw_vs_num_peaks_random",
)

In [ ]:
# Random split: peaks > 0.1 vs. num peaks, train/val/test
_shared_max_random = max(
    nist_train_stats_random["num_peaks"].max(),
    nist_val_stats_random["num_peaks"].max(),
    nist_test_stats_random["num_peaks"].max(),
)
plot_marginal_histograms(
    series_dict={
        "Train": nist_train_stats_random,
        "Val": nist_val_stats_random,
        "Test": nist_test_stats_random,
    },
    sample=[5000, None, 5000],
    x_key="num_peaks_above_01",
    y_key="num_peaks",
    x_label="Number of Peaks > 0.1",
    y_label="Number of Peaks",
    bins=40,
    alpha=0.5,
    s=5,
    colors=[palette[0], palette[3], palette[6]],
    legend_loc="upper left",
    legend_bbox=(0, 1.0),
    x_lim=(0, _shared_max_random),
    y_lim=(0, _shared_max_random),
    save_name="marginal_peaks_above_01_vs_num_peaks_random",
)